# <center>Variational Monte Carlo with Machine Learning Enhancement</center>
## <center>Alexander Yue (alexyue) - Physics 113 Final Project</center>

---

### <span style="color: #2E86AB; font-size: 18px;">Abstract</span>

<div style="background-color: #f8f9fa; padding: 15px; border-left: 4px solid #2E86AB; margin: 10px 0;">
In this project we use standard <strong>Variational Monte Carlo (VMC)</strong> to calculate the ground state energy of H₂ and LiH, then demonstrate recent developments that use machine learning to improve these calculations by using a neural network to represent <strong>wavefunctions</strong>.
</div>

---

### <span style="color: #2E86AB; font-size: 18px;">Introduction</span>

<div style="background-color: #fff3cd; padding: 10px; border-radius: 5px; margin: 10px 0;">
<strong>"The Big Picture":</strong><br>
A fundamental problem in condensed matter physics, chemistry, and materials science is determining the ground state energy of a molecule. This is a really hard problem. The Schrödinger equation for many electrons is not analytically solvable. And numerical solutions will have terrible exponential scaling. So what can we do?
</div>

---

### <span style="color: #2E86AB; font-size: 18px;">Background Research</span>

I used to work at a condensed matter computational physics lab, and my professor there recently published a paper on this subject, so I was inspired to learn how VMC works, how machine learning is being applied to it, and try it out myself with the help of some existing python libraries.

<div style="margin: 15px 0;">
<strong>Key Papers Studied:</strong>
<ul>
<li>"Autoregressive neural quantum states of Fermi Hubbard models" <a href="https://link.aps.org/doi/10.1103/PhysRevResearch.7.013122">https://link.aps.org/doi/10.1103/PhysRevResearch.7.013122</a></li>
<li>"Solving the quantum many-body problem with artificial neural networks" <a href="https://www.science.org/doi/abs/10.1126/science.aag2302">https://www.science.org/doi/abs/10.1126/science.aag2302</a></li>
</ul>
</div>

<div style="margin: 15px 0;">
<strong>Additional Resources:</strong>
<ul>
<li><a href="https://en.wikipedia.org/wiki/Variational_Monte_Carlo">Variational Monte Carlo (Wikipedia)</a></li>
<li><a href="https://en.wikipedia.org/wiki/Neural_network_quantum_states">Neural Network Quantum States (Wikipedia)</a></li>
<li><a href="https://netket.readthedocs.io/en/latest/tutorials/vmc-from-scratch.html">NetKet VMC Tutorial</a></li>
</ul>
</div>

After I gained a strong understanding of how VMC and ML-VMC worked, I began trying to write code to replicate the process. However, I realized it would be too time consuming to build such algorithms from scratch, and so I searched for existing packages and github repositories to utilize.

<div style="background-color: #e8f5e8; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Libraries Used:</strong>
<ul>
<li><strong>PySCF library</strong> for quantum chemistry (<a href="https://pyscf.org/">https://pyscf.org/</a>)</li>
<li><strong>PyQMC library</strong> for VMC functions (<a href="https://github.com/WagnerGroup/pyqmc">https://github.com/WagnerGroup/pyqmc</a>)</li>
<li><strong>DeepQMC library</strong> for ML-VMC functions (<a href="https://github.com/deepqmc/deepqmc">https://github.com/deepqmc/deepqmc</a>)</li>
</ul>
</div>

Since these methods, especially the ML-VMC method, are recent in development, the packages are not widely supported. Setting up dependencie without incompatibilities is tricky (this was the most challenging part of the research process). A working setup I found is documented in the README.

After following a few tutorials, I put everything together into one notebook, fixed a lot of bugs and got results! The molecules I chose to analyze are H₂ and LiH for their simplicity. While my implementations of VMC and ML-VMC both achieve accurate results, these molecules are not large enough to see the advantages of ML-VMC over traditional VMC. 

Future extrapolations of this work should include significantly more computational power and time, such that these advantages can be clearly seen.

---

### <span style="color: #2E86AB; font-size: 18px;">Traditional VMC</span>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Traditional Variational Monte Carlo:</strong><br>
The core idea is that we can start with a trial wavefunction (a guess, or "ansatz") and vary it, trying to find wavefunctions with lower energies. Once we cannot find a lower energy, we call it the ground state energy.
</div>

But how can we compute the energy of our guesses? We would have to compute: **E = ⟨ψ|H|ψ⟩/⟨ψ|ψ⟩**, but ψ is a complex superposition of all the base electron configurations. The amount of states in this basis will grow exponentially with the amount of electrons in our system. This is numerically intractable.

<strong>Instead, we can use the Monte Carlo method for evaluating the energy.</strong>

A wavefunction is a complex superposition of many base electron configurations. We can get the probability distribution of these configurations by if we had ψ². We can sample from this distribution and calculate the energy of it (easy to do for a single setup, just plug into the hamiltonian and do matrix multiplications). We can average these to get the energy of ψ. To sample efficiently, we use the metropolis, or "Monte Carlo" method where we sample configurations proportional to how probable they are. This is done with a series of proposed changes which are accepted or rejected randomly with acceptance probability **(ψ² new state / ψ² old state)**. Then we can do an unweighted sum of the states we visited to get the energy of this wavefunction.

In order to generate and vary guess wavefunctions, we need to define them in a parametrizable way. The first step is running the Hartree-Fock method (where we approximate that each electron acts as an independent orbital), which determines the molecular orbital structure of the molecule (more realistic than symmetric chemistry orbitals that don't take nuclei into account), but not the correlation between electrons (they avoid each other).

We then use the **Slater-Jastrow function** as our way to generate wavefunctions. The Slater part handles the Pauli exclusion principle, and the Jastrow factor handles the electron correlations. We can generate new wavefunctions just by varying the parameters in this function. (This actually does not cover all possible wavefunctions, but in practice this usually good enough, but more on this later)

---

### <span style="color: #2E86AB; font-size: 18px;">ML-VMC</span>

<div style="background-color: #ffebee; padding: 15px; border-left: 4px solid #d32f2f; margin: 15px 0;">
<strong>Key Drawback of Traditional VMC:</strong><br>
One key drawback of the traditional VMC method is the way we create and vary wavefunctions.
</div>

As you recall, we ran Hartree-Fock to compute the molecular orbitals, and then used the Slater-Jastrow function to generate wavefunctions. Not only is the Hartree-Fock approximation rather computationally expensive, the Slater-Jastrow function can only create a subset of possible wavefunctions. We have to hope that sampling these Slater-Jastrow functions is good enough as sampling the real wavefunctions.

<div style="background-color: #e8f5e8; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>The Neural Network Solution:</strong><br>
A more recent approach is to use a deep neural network to represent a wavefunction. This way, instead of only having a few parameters and a simplistic function to create wavefunctions, we can create immense complexity and be more representative of the true space of possible wavefunctions.
</div>

The **"Neural Ansatz"** in particular is a Graph Neural Network, because the electrons and their interactions are a graph. It has some special rules to make it Slater-Jastrow-like, meaning it will automatically enforce properties like Pauli exclusion (no use exploring wavefunctions that can't exist), but it has more potential complexity.

<div style="margin: 15px 0;">
<strong>ML-VMC Process:</strong>
<ol>
<li>Initialize a neural ansatz with random weights</li>
<li>Take samples with metropolis again</li>
<li>Pick a basis state and pass it through our neural wavefunction</li>
<li>Square this result to get ψ², needed for metropolis</li>
<li>Calculate the energy of this basis state directly</li>
<li>Treat energy as a loss function and backpropagate</li>
</ol>
</div>

While we are evaluating the energy of a state on our ansatz, we can treat it as a loss function and backpropagate, so the ML model will learn to minimize it. And we are looking for the lowest energy so this is perfect! (But we do have to be careful about creating training batches from varied basis states)

Also, because the loss function here is very complex and has some randomness, we don't want to do normal gradient descent with ADAM. What works better is **Kronecker-Factored Approximate Curvature (KFAC)**.

Also it does help to pre-train the model on a bunch of Hartree-Fock results so it "gets its bearings" ahead of time, and avoid local minima.

<div style="background-color: #fff3cd; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Benefits of ML-VMC:</strong>
<ul>
<li>Better accuracy in ground state energy by sampling from a more complete set of wavefunctions</li>
<li>Better scaling with number of electrons</li>
<li>Seems to converge in fewer iterations</li>
<li>ML models let us take advantage of GPUs for faster compute</li>
</ul>
</div>

---

### <span style="color: #2E86AB; font-size: 18px;">Methods</span>

<div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin: 15px 0;">
A high level description of the code below is as follows (inline comments and cell markdown explanations are provided as well):
</div>

<div style="margin: 15px 0;">
<strong>Traditional VMC Process:</strong>
<ol>
<li>Get our molecular system - generate a Hamiltonian for LiH using PySCF through the powerful <code>Mole()</code> function</li>
<li>Run Hartree-Fock algorithm - gives a good starting guess for the ground state energy</li>
<li>Optimize the guess with the pyqmc library (VMC) using the Jastrow factor</li>
<li>Take a final measurement of the ground state energy with Monte Carlo sampling</li>
<li>Calculate correlation energy to quantify success</li>
</ol>
</div>

<div style="background-color: #e3f2fd; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Results - Traditional VMC:</strong>
<ul>
<li><strong>H₂:</strong> Correlation energy of <strong>-0.095170 Ha</strong></li>
<li><strong>LiH:</strong> Correlation energy of <strong>-0.173789 Ha</strong></li>
</ul>
</div>

<div style="margin: 15px 0;">
<strong>ML-VMC Process:</strong>
<ol>
<li>Setup molecules using PySCF as before</li>
<li>Set up a neural ansatz and a KFAC optimizer with deepqmc</li>
<li>Train for approximately 5 minutes each (no GPU acceleration available)</li>
</ol>
</div>

<div style="background-color: #fff3cd; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Results - ML-VMC:</strong>
<ul>
<li><strong>H₂:</strong> Correlation energy of <strong>-0.057906 Ha</strong></li>
<li><strong>LiH:</strong> Correlation energy of <strong>-0.081788 Ha</strong></li>
</ul>
<em>Note: ML-VMC results are lower than Traditional VMC, but would likely improve given adequate training time.</em>
</div>

---

### <span style="color: #2E86AB; font-size: 18px;">Works Cited</span>

<div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Primary Libraries:</strong>
</div>

This project would not be possible without use of the incredible **PySCF library** for quantum chemistry (https://pyscf.org/)

I followed and used code from the quickstart tutorial from their official docs (https://pyscf.org/quickstart.html)

In addition, for the implementation of VMC, the **PyQMC library** was invaluable (https://github.com/WagnerGroup/pyqmc)

I followed and used code from their "recipe book" of useful functions for VMC using PySCF (https://pyqmc.readthedocs.io/en/latest/tutorial.html)

Furthermore, for the ML-enhancement of VMC the library **DeepQMC** was used (https://github.com/deepqmc/deepqmc)

I followed and used code from their official tutorial (https://deepqmc.github.io/tutorial.html)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>Additional Inspiration and Reference Sources:</strong>
<ul>
<li>https://netket.readthedocs.io/en/latest/tutorials/vmc-from-scratch.html</li>
<li>https://github.com/google-deepmind/ferminet</li>
</ul>
</div>

---

### <span style="color: #2E86AB; font-size: 18px;">References</span>

```bibtex
@article{PhysRevResearch.7.013122,
  title = {Autoregressive neural quantum states of Fermi Hubbard models},
  author = {Ibarra-Garc\'{\i}a-Padilla, Eduardo and Lange, Hannah and Melko, Roger G. and Scalettar, Richard T. and Carrasquilla, Juan and Bohrdt, Annabelle and Khatami, Ehsan},
  publisher = {American Physical Society},
  doi = {10.1103/PhysRevResearch.7.013122},
  url = {https://link.aps.org/doi/10.1103/PhysRevResearch.7.013122}
}

@article{
doi:10.1126/science.aag2302,
author = {Giuseppe Carleo  and Matthias Troyer },
title = {Solving the quantum many-body problem with artificial neural networks},
journal = {Science},
doi = {10.1126/science.aag2302},
URL = {https://www.science.org/doi/abs/10.1126/science.aag2302}
}

#### Setup

In [1]:
# First lets set up our important imports and confirm they are working
print("=" * 50 + "\nTesting Key Imports for VMC\n" + "=" * 50)
    
try: import pyscf; print("PySCF version:", pyscf.__version__)
except ImportError as e: print("PySCF import failed:", e)
try: import pyqmc; print("PyQMC imported successfully")
except ImportError as e: print("PyQMC import failed:", e)
try: import deepqmc; print("DeepQMC imported successfully")  
except ImportError as e: print("DeepQMC import failed:", e)
try: import numpy as np; print("NumPy version:", np.__version__)
except ImportError as e: print("NumPy import failed:", e)

try: 
    import torch
    cuda_info = f", CUDA: {torch.cuda.is_available()}"
    if torch.cuda.is_available(): cuda_info += f" ({torch.cuda.device_count()} devices)"
    print(f"PyTorch version: {torch.__version__}{cuda_info}")
except ImportError as e: print("PyTorch import failed:", e)

print("=" * 50 + "\nImport testing complete!\n" + "=" * 50)

Testing Key Imports for VMC
PySCF version: 2.10.0
PyQMC imported successfully
DeepQMC imported successfully
NumPy version: 1.26.4
PyTorch version: 2.7.1, CUDA: False
Import testing complete!


In [ ]:
# This setup is needed for the PySCF code
def setup_environment():
    import os
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1" 
    os.environ["OMP_NUM_THREADS"] = "1"
    
    for fname in ['mf.hdf5','optimized_wf.hdf5','vmc_data.hdf5','dmc.hdf5']:
        if os.path.isfile(fname): os.remove(fname)
    
    print("Environment configured: single-threaded execution")
    print("Previous calculation files cleaned up")
    
setup_environment()

#### First we need to set up our H2 molecule with the PySCF library
This follows the Born-Oppenheimer approximation where we fix the positions of nuclei

The Electrons are defined in 3d by three gaussians

In [ ]:
def create_h2_system():
    from pyscf import gto
    print("=" * 40 + "\nH2 Molecular System Setup\n" + "=" * 40)
    
    # Define H2 geometry, three dimensions
    mol = gto.M(
        atom=[['H', (0, 0, 0)], ['H', (0, 0, 1.4)]],  # This is specifying length of the bond between electrons
        basis='sto-3g'
    )
    
    print(f"H2 molecule created")
    print(f"  Atoms: {mol.natm}")
    print(f"  Electrons: {mol.nelectron}")
    print(f"  Bond length: 1.4 bohr (≈ {1.4 * 0.529:.2f} Angstrom)")
    print(f"  Basis set: STO-3G")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    
    return mol

h2_mol = create_h2_system()

#### Then, we can run Hartree Fock 

Hartree Fock is a mean field calculation

It assumes each electron is represented by a average of a field, ignoring electron repulsion

This allows us to quicky get a base guess for the energy

In [ ]:
def Hartree_Fock_calculation(mol):
    from pyscf import scf
    print("=" * 40 + "\nH2 Mean-Field Calculation\n" + "=" * 40)
        
    # Run Hartree-Fock with pyscf build in function RHF
    mf = scf.RHF(mol)
    mf.chkfile = "mf.hdf5"
    energy = mf.kernel()
    
    print(f"HF calculation converged: E = {energy:.8f} Ha")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print(f"  Electronic energy: {energy - mol.energy_nuc():.6f} Ha")
    print("Results saved to mf.hdf5")
    
    return mf

h2_mf = Hartree_Fock_calculation(h2_mol)

#### Now we optimize for the wavefunction with lowest energy

pyqmc has a built in function for this

It uses the Slater Jastrow function for wavefunctions 

In [ ]:
def wavefunction_optimization():
    """Optimization: Optimize Jastrow parameters using VMC"""
    import pyqmc.api as pyq
    print("=" * 40 + "\nWave Function Optimization\n" + "=" * 40)
    
    # Optimize with Slater-Jastrow wave function
    pyq.OPTIMIZE("mf.hdf5", "optimized_wf.hdf5", nconfig=100, max_iterations=10, verbose=True)
    
    # Read and display results
    df = pyq.read_opt("optimized_wf.hdf5")
    initial_e = df.iloc[0]['energy']
    final_e = df.iloc[-1]['energy'] 
    final_err = df.iloc[-1]['error']
    
    print(f"Optimization completed: {len(df)} iterations")
    print(f"  Initial energy: {initial_e:.6f} Ha")
    print(f"  Final energy: {final_e:.6f} +/- {final_err:.6f} Ha")
    print(f"  Energy lowering: {initial_e - final_e:.6f} Ha")
    return df

opt_df = wavefunction_optimization()

#### Finally, lets measure the final wavefunction energy with higher precision with more Monte Carlo Sampling

With this, we can draw our final results

In [ ]:
def vmc_sampling():
    """VMC: Sample optimized wave function for accurate energy"""
    import pyqmc.api as pyq
    print("=" * 40 + "\nVMC Energy Sampling\n" + "=" * 40)
    
    # Run VMC with our optimized parameters
    pyq.VMC("mf.hdf5", "vmc_data.hdf5", load_parameters="optimized_wf.hdf5", 
            nblocks=30, verbose=False)
    
    # display results
    results = pyq.read_mc_output("vmc_data.hdf5")
    
    print(f"VMC completed successfully")
    print(f"  Total energy: {results['energytotal']:.8f} +/- {results['energytotal_err']:.8f} Ha")
    print(f"  Kinetic energy: {results['energyke']:.6f} +/- {results['energyke_err']:.6f} Ha")
    print(f"  Potential energy: {results['energytotal'] - results['energyke']:.6f} Ha")
    if 'acceptance' in results:
        print(f"  Acceptance rate: {results['acceptance']:.1%}")
    print(f"  Available data keys: {list(results.keys())}")
    return results

vmc_results = vmc_sampling()

#### Correlation Energy Recovery

The correlation energy shows how much of the energy difference VMC recovers beyond what we got from Hartree-Fock

In [ ]:
def analyze_correlation_energy(vmc_results, hf_results, molecule_name="H2"):
    """Analysis: Calculate and display correlation energy recovery"""
    print("=" * 40 + f"\n{molecule_name} Correlation Analysis\n" + "=" * 40)
    
    # Extract energies
    hf_energy = hf_results.e_tot
    vmc_energy = vmc_results['energytotal']
    vmc_error = vmc_results['energytotal_err']
    
    # Calculate correlation energy (energy lowering from HF to VMC)
    correlation_energy = vmc_energy - hf_energy
    
    print(f"Hartree-Fock energy: {hf_energy:.8f} Ha")
    print(f"VMC energy: {vmc_energy:.8f} +/- {vmc_error:.8f} Ha")
    print(f"Correlation energy: {correlation_energy:.6f} Ha")
    
    # Convert to more intuitive units
    correlation_kcal = correlation_energy * 627.509  # Ha to kcal/mol
    print(f"Correlation energy: {correlation_kcal:.2f} kcal/mol")
    
    # Analysis of the result
    if abs(correlation_energy) > 0.01:  # Significant correlation
        print(f"VMC successfully recovered {abs(correlation_energy):.6f} Ha of correlation energy")
        print("This represents the energy gained by allowing electrons to correlate their motion (interact with each other)")
    else:
        print("Minimal correlation energy recovered - system may be well-described by HF")
    
    return correlation_energy

h2_corr_energy = analyze_correlation_energy(vmc_results, h2_mf, "H2")


### Now lets do it all again but with LiH

LiH is hetronuclear, meaning there will be an asymmetric charge distribution

It has 4 electrons, a good test of increased complexity

In [ ]:
def create_lih_system():
    from pyscf import gto
    print("=" * 40 + "\nLiH Molecular System Setup\n" + "=" * 40)
    
    atoms = [['Li', (0, 0, 0)], ['H', (0, 0, 3.0)]]
    
    mol = gto.M(atom=atoms, basis='sto-3g')
    
    print(f"LiH molecule created")
    print(f"  Atoms: {mol.natm} ({mol.atom_symbol(0)}, {mol.atom_symbol(1)})")
    print(f"  Electrons: {mol.nelectron}")
    print(f"  Bond length: 3.0 bohr (≈ {3.0 * 0.529:.2f} Å)")
    print(f"  Basis set: STO-3G")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print("  Electronic structure: Li 1s²2s¹ + H 1s¹ → Li⁺H⁻-like bonding")
    
    return mol

def lih_Hartree_Fock_calculation(mol):
    from pyscf import scf
    print("=" * 40 + "\nLiH Mean-Field Calculation\n" + "=" * 40)
    
    print("[STARTING] LiH Hartree-Fock calculation...")
    print("  Challenge: 4 electrons, heteronuclear bonding")
    
    mf = scf.RHF(mol)
    mf.chkfile = 'lih_mf.hdf5'
    energy = mf.kernel()
    
    print(f"HF calculation converged: E = {energy:.8f} Ha")
    print(f"  Nuclear repulsion: {mol.energy_nuc():.6f} Ha")
    print(f"  Electronic energy: {energy - mol.energy_nuc():.6f} Ha")
    print(f"Results saved to lih_mf.hdf5")
    
    return mf

def lih_optimization():
    import pyqmc.api as pyq
    print("=" * 40 + "\nLiH Wave Function Optimization\n" + "=" * 40)
    
    pyq.OPTIMIZE("lih_mf.hdf5", "lih_optimized_wf.hdf5", nconfig=100, max_iterations=10, verbose=False)
    
    df = pyq.read_opt("lih_optimized_wf.hdf5")
    initial_e = df.iloc[0]['energy']
    final_e = df.iloc[-1]['energy'] 
    final_err = df.iloc[-1]['error']
    
    print(f"LiH optimization completed: {len(df)} iterations")
    print(f"  Initial energy: {initial_e:.6f} Ha")
    print(f"  Final energy: {final_e:.6f} +/- {final_err:.6f} Ha")
    print(f"  Energy lowering: {initial_e - final_e:.6f} Ha")
    return df

def lih_vmc_sampling():
    import pyqmc.api as pyq
    print("=" * 40 + "\nLiH VMC Energy Sampling\n" + "=" * 40)
    
    pyq.VMC("lih_mf.hdf5", "lih_vmc_data.hdf5", load_parameters="lih_optimized_wf.hdf5", 
            nblocks=30, verbose=False)
    
    results = pyq.read_mc_output("lih_vmc_data.hdf5")
    
    print(f"LiH VMC completed successfully")
    print(f"  Total energy: {results['energytotal']:.8f} +/- {results['energytotal_err']:.8f} Ha")
    print(f"  Kinetic energy: {results['energyke']:.6f} +/- {results['energyke_err']:.6f} Ha")
    print(f"  Potential energy: {results['energytotal'] - results['energyke']:.6f} Ha")
    if 'acceptance' in results:
        print(f"  Acceptance rate: {results['acceptance']:.1%}")
    return results


def comparison_summary(h2_results, lih_results, h2_mf, lih_mf):
    print("=" * 50 + "\nH2 vs LiH Comparison Summary\n" + "=" * 50)
    
    print("Hartree-Fock Energies:")
    print(f"  H2:  {h2_mf.e_tot:.8f} Ha")
    print(f"  LiH: {lih_mf.e_tot:.8f} Ha")
    
    print(f"\nVMC Energies:")
    print(f"  H2:  {h2_results['energytotal']:.8f} +/- {h2_results['energytotal_err']:.8f} Ha")
    print(f"  LiH: {lih_results['energytotal']:.8f} +/- {lih_results['energytotal_err']:.8f} Ha")
    
    h2_corr = h2_results['energytotal'] - h2_mf.e_tot
    lih_corr = lih_results['energytotal'] - lih_mf.e_tot
    print(f"\nElectron Correlation Energy (VMC - HF):")
    print(f"  H2:  {h2_corr:.6f} Ha")
    print(f"  LiH: {lih_corr:.6f} Ha")
    
    print("=" * 50 + "\nComparison complete!\n" + "=" * 50)

lih_mol = create_lih_system()
lih_mf = lih_Hartree_Fock_calculation(lih_mol)
lih_opt_df = lih_optimization()
lih_vmc_results = lih_vmc_sampling()
lih_corr_energy = analyze_correlation_energy(lih_vmc_results, lih_mf, "LiH")

### Now lets compare our results for H2 and LiH

In [ ]:
def comparison_summary(h2_results, lih_results, h2_mf, lih_mf):
    """Summary: Compare H2 vs LiH VMC results"""
    print("=" * 50 + "\nH2 vs LiH Comparison Summary\n" + "=" * 50)
    
    # HF energies comparison
    print("Hartree-Fock Energies:")
    print(f"  H2:  {h2_mf.e_tot:.8f} Ha")
    print(f"  LiH: {lih_mf.e_tot:.8f} Ha")
    
    # VMC energies comparison
    print(f"\nVMC Energies:")
    print(f"  H2:  {h2_results['energytotal']:.8f} +/- {h2_results['energytotal_err']:.8f} Ha")
    print(f"  LiH: {lih_results['energytotal']:.8f} +/- {lih_results['energytotal_err']:.8f} Ha")
    
    # Correlation energy (VMC - HF)
    h2_corr = h2_results['energytotal'] - h2_mf.e_tot
    lih_corr = lih_results['energytotal'] - lih_mf.e_tot
    print(f"\nElectron Correlation Energy (VMC - HF):")
    print(f"  H2:  {h2_corr:.6f} Ha")
    print(f"  LiH: {lih_corr:.6f} Ha")
    
    print("=" * 50)

comparison_summary(vmc_results, lih_vmc_results, h2_mf, lih_mf)

## Now that we have seen the utility of traditinal VMC, lets see how ML can enhance it

#### Setup

In [ ]:
def test_ml_imports():
    """Test: Verify all ML-VMC dependencies are working"""
    print("=" * 50 + "\nTesting ML-VMC Imports\n" + "=" * 50)
    
    try: import jax; print("JAX version:", jax.__version__)
    except ImportError as e: print("ERROR: JAX import failed:", e)
    try: import haiku; print("Haiku version:", haiku.__version__)
    except ImportError as e: print("ERROR: Haiku import failed:", e)
    try: import deepqmc; print("DeepQMC imported successfully")
    except ImportError as e: print("ERROR: DeepQMC import failed:", e)
    try: import hydra; print("Hydra version:", hydra.__version__)
    except ImportError as e: print("ERROR: Hydra import failed:", e)
    
    try:
        import jax
        devices = jax.devices()
        device_info = f"{len(devices)} device(s): {[d.device_kind for d in devices]}"
        print(f"JAX devices: {device_info}")
        if any(d.device_kind == 'gpu' for d in devices):
            print("[ACCELERATED] GPU acceleration available!")
        else:
            print("[INFO] Running on CPU (GPU acceleration requires CUDA-enabled JAX)")
    except Exception as e: print("[WARNING] JAX device info failed:", e)
    
    print("=" * 50 + "\nML import testing complete!\n" + "=" * 50)

test_ml_imports()

In [ ]:
def setup_ml_environment():
    """Setup: Configure JAX and clean up old ML runs"""
    import os
    import shutil
    
    # JAX configuration for reproducibility
    os.environ["JAX_ENABLE_X64"] = "True"  # Double precision
    
    # Clean up any existing ML runs
    for dirname in ['h2_ml_runs', 'lih_ml_runs', 'runs']:
        if os.path.exists(dirname): shutil.rmtree(dirname)
    
    print("JAX configured: double precision enabled")
    print("Previous ML calculation directories cleaned up")

setup_ml_environment()

#### We set up our H2 molecule with PySCF again and get its Hamiltonian 

This allows us to get the energy of a specific basis state

In [ ]:
def create_h2_molecule():
    from deepqmc.molecule import Molecule
    from deepqmc.hamil import MolecularHamiltonian
    print("=" * 40 + "\nH2 Molecule Setup (ML-VMC)\n" + "=" * 40)
    
    # Create H2 molecule (same geometry as traditional VMC)
    mol = Molecule(
        coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 0.74]], 
        charges=[1, 1], 
        charge=0, 
        spin=0, 
        unit='angstrom'
    )
    
    print(f"H2 molecule created: bond length = 0.74 A")
    print(f"  Coordinates: {mol.coords}")
    print(f"  Charges: {mol.charges}, Total charge: {mol.charge}, Spin: {mol.spin}")
    
    # Create Hamiltonian
    H = MolecularHamiltonian(mol=mol)
    print("Molecular Hamiltonian created")
    print("Ready for neural wave function setup")
    
    return mol, H

mol_h2, H_h2 = create_h2_molecule()

#### Setup neural network wavefunction

This uses a Graph Neural Network instead of the Slater Jastrow function

The GNN allows us to represent much more complex wavefunctions

Some physical laws are enforced on the GNN so we cannot create physically impossible wavefunctions

In [ ]:
def setup_neural_ansatz(H):
    import os
    import haiku as hk
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    from deepqmc.app import instantiate_ansatz
    import deepqmc
    
    print("=" * 40 + "\nNeural Wave Function Setup\n" + "=" * 40)
    
    # Load default neural ansatz configuration
    deepqmc_dir = os.path.dirname(deepqmc.__file__)
    config_dir = os.path.join(deepqmc_dir, 'conf/ansatz')
    
    with initialize_config_dir(version_base=None, config_dir=config_dir):
        cfg = compose(config_name='default')
    
    print("Loaded default ansatz configuration")
    print(f"  Config type: {cfg._target_}")
    
    # Instantiate the neural ansatz
    _ansatz = instantiate(cfg, _recursive_=True, _convert_='all')
    ansatz = instantiate_ansatz(H, _ansatz)
    
    print("Neural wave function ansatz created")
    print("  Architecture: Graph Neural Network with electron embeddings")
    print("  Type: Slater-Jastrow-like with learned orbitals")
    
    return ansatz

ansatz_h2 = setup_neural_ansatz(H_h2)

#### Setup enhanced Monte Carlo sampler for neural networks

In [ ]:
def setup_ml_sampler():
    from deepqmc.sampling import initialize_sampling, combine_samplers, MetropolisSampler, DecorrSampler
    from functools import partial
    
    print("=" * 40 + "\nML-Enhanced Sampler Setup\n" + "=" * 40)
    
    # Create combined sampler (Metropolis + decorrelation) 
    elec_sampler = partial(combine_samplers, samplers=[
        DecorrSampler(length=10),
        MetropolisSampler
    ])
    sampler_factory = partial(initialize_sampling, elec_sampler=elec_sampler)
    
    print("Advanced sampler factory created")
    print("  Components: DecorrSampler(10) + MetropolisSampler") 
    print("  Features: Automatic decorrelation + adaptive step size")
    
    return sampler_factory

sampler_factory = setup_ml_sampler()

#### Setup KFAC optimizer for neural training

KFAC is better than ADAM for neural wavefunctions

In [ ]:
def setup_optimizer():
    import os
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    import deepqmc
    
    print("=" * 40 + "\nKFAC Optimizer Setup\n" + "=" * 40)
    
    # Load KFAC optimizer configuration
    deepqmc_dir = os.path.dirname(deepqmc.__file__)
    opt_config_dir = os.path.join(deepqmc_dir, 'conf/task/opt')
    
    with initialize_config_dir(version_base=None, config_dir=opt_config_dir):
        opt_cfg = compose(config_name='kfac')
    
    optimizer = instantiate(opt_cfg, _recursive_=True, _convert_='all')
    
    print("KFAC optimizer loaded")
    print("  Type: Kronecker-Factored Approximate Curvature")
    print("  Features: Second-order optimization for neural networks")
    
    return optimizer

optimizer = setup_optimizer()

In [ ]:
#### Train the neural wavefunction on H2

def train_h2_neural_wf(H, ansatz, optimizer, sampler_factory):
    from deepqmc.train import train
    print("=" * 40 + "\nH2 Neural Wave Function Training\n" + "=" * 40)
    
    print("Neural wave function training...")
    print("  System: H2 molecule")
    print("  Method: Variational Monte Carlo with neural ansatz")
    
    # Run training with progress monitoring
    try:
        train(
            H, ansatz, optimizer, sampler_factory,
            steps=500,                   
            electron_batch_size=500,     
            seed=42,                      
            workdir='h2_ml_runs',
            pretrain_steps=100,
            max_restarts=2
        )
        print("Training completed successfully!")
        
    except Exception as e:
        print(f"Training encountered issue: {e}")
        print("  This is normal for first-time ML-VMC runs")
    
    print("Results saved to h2_ml_runs/ directory")

train_h2_neural_wf(H_h2, ansatz_h2, optimizer, sampler_factory)

#### Analyze H2 ML-VMC results

In [ ]:
def analyze_h2_results():
    import h5py
    import numpy as np
    import os
    import matplotlib.pyplot as plt
    
    print("=" * 40 + "\nH2 ML-VMC Results Analysis\n" + "=" * 40)
    
    results_file = 'h2_ml_runs/training/result.h5'
    
    if not os.path.exists(results_file):
        print("Results file not found - training may not have completed")
        return None
    
    with h5py.File(results_file, 'r') as f:
        print("Results file found, analyzing data...")
        print(f"  Available data: {list(f.keys())}")
        
        if 'local_energy' in f:
            energy_data = f['local_energy']
            print(f"  Energy data keys: {list(energy_data.keys())}")
            
            if 'mean' in energy_data:
                energy_means = np.array(energy_data['mean']).flatten()
                energy_stds = np.array(energy_data['std']).flatten() if 'std' in energy_data else np.zeros_like(energy_means)
                steps = np.arange(len(energy_means))
                
                final_energy = energy_means[-1]
                energy_std = energy_stds[-1]
                
                print(f"Final ML-VMC energy: {final_energy:.8f} +/- {energy_std:.8f} Ha")
                print(f"  Training convergence: {len(energy_means)} steps")
                
                try:
                    plt.figure(figsize=(12, 5))
                    
                    plt.subplot(1, 2, 1)
                    plt.plot(steps, energy_means, 'b-', linewidth=1.5, alpha=0.8, label='Energy')
                    if np.any(energy_stds > 0):
                        plt.fill_between(steps, energy_means - energy_stds, energy_means + energy_stds, 
                                       alpha=0.3, label='±1σ uncertainty')
                    plt.axhline(y=final_energy, color='r', linestyle='--', alpha=0.7, label=f'Final: {final_energy:.6f} Ha')
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy (Ha)')
                    plt.title('H₂ ML-VMC Energy Convergence')
                    plt.legend()
                    plt.grid(True, alpha=0.3)
                    
                    plt.subplot(1, 2, 2)
                    initial_energy = energy_means[0]
                    energy_improvement = (energy_means - initial_energy) * 1000
                    plt.plot(steps, energy_improvement, 'g-', linewidth=1.5)
                    plt.axhline(y=0, color='k', linestyle=':', alpha=0.5)
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy Change (mHa)')
                    plt.title('H₂ Energy Improvement from Initial')
                    plt.grid(True, alpha=0.3)
                    
                    plt.tight_layout()
                    
                    os.makedirs('h2_ml_runs', exist_ok=True)
                    plt.savefig('h2_ml_runs/h2_energy_convergence.png', dpi=300, bbox_inches='tight')
                    print("  ✓ Energy convergence plot saved: h2_ml_runs/h2_energy_convergence.png")
                    plt.show()
                    
                except Exception as e:
                    print(f"  ⚠ Could not create energy plots: {e}")
                
                return {'energy': final_energy, 'error': energy_std, 'training_steps': len(energy_means)}
    
    return None




h2_ml_result = analyze_h2_results()

#### Now do the same for LiH 

In [ ]:
def create_lih_molecule():
    from deepqmc.molecule import Molecule
    from deepqmc.hamil import MolecularHamiltonian
    print("=" * 40 + "\nLiH Molecule Setup (ML-VMC)\n" + "=" * 40)
    
    mol = Molecule(
        coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 1.6]], 
        charges=[3, 1], 
        charge=0, 
        spin=0, 
        unit='angstrom'
    )
    
    print(f"LiH molecule created: Li-H bond length = 1.6 A")
    print(f"  4 electrons: more challenging than H2")
    print(f"  Charges: Li(+3), H(+1), Total charge: {mol.charge}")
    
    H = MolecularHamiltonian(mol=mol, elec_std=0.8)
    print("LiH Hamiltonian created with adjusted electron distribution")
    
    return mol, H

def train_lih_neural_wf(H, sampler_factory):
    from deepqmc.train import train
    import os
    from hydra import compose, initialize_config_dir
    from hydra.utils import instantiate
    from deepqmc.app import instantiate_ansatz
    import deepqmc
    
    print("=" * 40 + "\nLiH Neural Wave Function Training\n" + "=" * 40)
    
    deepqmc_dir = os.path.dirname(deepqmc.__file__)
    
    config_dir = os.path.join(deepqmc_dir, 'conf/ansatz')
    with initialize_config_dir(version_base=None, config_dir=config_dir):
        cfg = compose(config_name='default')
    _ansatz = instantiate(cfg, _recursive_=True, _convert_='all')
    ansatz = instantiate_ansatz(H, _ansatz)
    
    # Optimizer
    opt_config_dir = os.path.join(deepqmc_dir, 'conf/task/opt')
    with initialize_config_dir(version_base=None, config_dir=opt_config_dir):
        opt_cfg = compose(config_name='kfac')
    optimizer = instantiate(opt_cfg, _recursive_=True, _convert_='all')
    
    print("LiH neural wave function training...")
    print("  System: LiH molecule (4 electrons)")
    print("  Challenge: Larger system than H2")
    
    try:
        train(
            H, ansatz, optimizer, sampler_factory,
            steps=500,
            electron_batch_size=300,
            seed=42,
            workdir='lih_ml_runs',
            pretrain_steps=100,
            max_restarts=3
        )
        print("LiH training completed successfully!")
        
    except Exception as e:
        print(f"LiH training encountered issue: {e}")
        print("  4-electron systems are more challenging")
    
    print("LiH results saved to lih_ml_runs/ directory")

def analyze_lih_results():
    import h5py
    import numpy as np
    import os
    import matplotlib.pyplot as plt
    
    print("=" * 40 + "\nLiH ML-VMC Results Analysis\n" + "=" * 40)
    
    results_file = 'lih_ml_runs/training/result.h5'
    
    if not os.path.exists(results_file):
        print("LiH results file not found")
        return None
    
    with h5py.File(results_file, 'r') as f:
        print("LiH results file found, analyzing data...")
        
        if 'local_energy' in f:
            energy_data = f['local_energy']
            
            if 'mean' in energy_data:
                energy_means = np.array(energy_data['mean']).flatten()
                energy_stds = np.array(energy_data['std']).flatten() if 'std' in energy_data else np.zeros_like(energy_means)
                steps = np.arange(len(energy_means))
                
                final_energy = energy_means[-1]
                energy_std = energy_stds[-1]
                
                print(f"Final LiH ML-VMC energy: {final_energy:.8f} +/- {energy_std:.8f} Ha")
                print(f"  Training steps: {len(energy_means)}")
                
                try:
                    plt.figure(figsize=(12, 5))
                    
                    plt.subplot(1, 2, 1)
                    plt.plot(steps, energy_means, 'b-', linewidth=1.5, alpha=0.8, label='Energy')
                    if np.any(energy_stds > 0):
                        plt.fill_between(steps, energy_means - energy_stds, energy_means + energy_stds, 
                                       alpha=0.3, label='±1σ uncertainty')
                    plt.axhline(y=final_energy, color='r', linestyle='--', alpha=0.7, label=f'Final: {final_energy:.6f} Ha')
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy (Ha)')
                    plt.title('LiH ML-VMC Energy Convergence')
                    plt.legend()
                    plt.grid(True, alpha=0.3)
                    
                    plt.subplot(1, 2, 2)
                    initial_energy = energy_means[0]
                    energy_improvement = (energy_means - initial_energy) * 1000
                    plt.plot(steps, energy_improvement, 'g-', linewidth=1.5)
                    plt.axhline(y=0, color='k', linestyle=':', alpha=0.5)
                    plt.xlabel('Training Step')
                    plt.ylabel('Energy Change (mHa)')
                    plt.title('LiH Energy Improvement from Initial')
                    plt.grid(True, alpha=0.3)
                    
                    plt.tight_layout()
                    
                    os.makedirs('lih_ml_runs', exist_ok=True)
                    plt.savefig('lih_ml_runs/lih_energy_convergence.png', dpi=300, bbox_inches='tight')
                    print("  ✓ Energy convergence plot saved: lih_ml_runs/lih_energy_convergence.png")
                    plt.show()
                    
                except Exception as e:
                    print(f"  ⚠ Could not create energy plots: {e}")
                
                return {'energy': final_energy, 'error': energy_std, 'training_steps': len(energy_means)}
    

mol_lih, H_lih = create_lih_molecule()
train_lih_neural_wf(H_lih, sampler_factory)
lih_ml_result = analyze_lih_results()

### Great! Lets compare this ML-VMC approach with our traditional VMC

In [ ]:
def ml_traditional_comparison(h2_ml_result, lih_ml_result, h2_corr_energy, lih_corr_energy):
    # Hartree-Fock reference energies (Ha)
    h2_hf = -1.1167 
    lih_hf = -7.9877

    h2_ml_corr = float(h2_ml_result['energy']) - h2_hf
    lih_ml_corr = float(lih_ml_result['energy']) - lih_hf
    
    print(f"{'Method':<20} {'Molecule':<8} {'E_corr (Ha)':<15} {'E_corr (mHa)':<15}")
    print(f"{'-'*60}")
    print(f"{'Traditional VMC':<20} {'H2':<8} {h2_corr_energy:<15.6f} {h2_corr_energy*1000:<15.1f}")
    print(f"{'Traditional VMC':<20} {'LiH':<8} {lih_corr_energy:<15.6f} {lih_corr_energy*1000:<15.1f}")
    print(f"{'ML-VMC (neural)':<20} {'H2':<8} {h2_ml_corr:<15.6f} {h2_ml_corr*1000:<15.1f}")
    print(f"{'ML-VMC (neural)':<20} {'LiH':<8} {lih_ml_corr:<15.6f} {lih_ml_corr*1000:<15.1f}")

ml_traditional_comparison(h2_ml_result, lih_ml_result, h2_corr_energy, lih_corr_energy) 

In [ ]:
h2_ml_result.__dict__